In [10]:
import os
import pandas as pd
from dotenv import load_dotenv
from opendartreader import OpenDartReader

# API 키 로드 및 DART 객체 생성
load_dotenv()
api_key = os.environ.get('DART_API_KEY')
dart = OpenDartReader(api_key)

# 매핑 테이블 불러오기
current_path = os.getcwd()
root_path = os.path.dirname(current_path)
csv_path = os.path.join(root_path, 'data', 'kospi_top50_mapping.csv')
mapping_df = pd.read_csv(csv_path, dtype={'corp_code': str})

# 테스트할 기업을 '카카오'로 설정하여 corp_code 추출
test_corp_name = '삼성전자'
raw_code = mapping_df.loc[mapping_df['corp_name'] == test_corp_name, 'corp_code'].values[0]
test_corp_code = str(raw_code).zfill(8)

print(f"✅ 초기 세팅 완료! 타겟 기업: {test_corp_name} (고유번호: {test_corp_code})")

✅ 초기 세팅 완료! 타겟 기업: 삼성전자 (고유번호: 00126380)


In [11]:
import pandas as pd
from datetime import datetime
from dateutil.relativedelta import relativedelta
import json

def calculate_total_risk(corp_name, corp_code, dart_api):
    """
    특정 기업의 재무 및 텍스트 공시 데이터를 종합하여 최종 리스크 점수와 등급을 반환합니다.
    """
    # API 응답용 데이터 구조
    result = {
        "corp_name": corp_name,
        "total_score": 0,
        "grade": "✅ 안전 (Safe)",
        "financial_risk": {"score": 0, "details": []},
        "text_risk": {"score": 0, "details": []}
    }
    
    # --- 재무 리스크 평가 (최근 사업년도 2025 기준) ---
    try:
        finstate = dart_api.finstate(corp_code, '2025', reprt_code='11011')
        if finstate is not None and not finstate.empty:
            def clean_amt(val): return float(str(val).replace(',', '')) if not pd.isna(val) else 0

            liab_row = finstate[finstate['account_nm'] == '부채총계']
            eqty_row = finstate[finstate['account_nm'] == '자본총계']
            op_row = finstate[finstate['account_nm'] == '영업이익']

            # 부채비율 계산
            if not liab_row.empty and not eqty_row.empty:
                liab = clean_amt(liab_row['thstrm_amount'].values[0])
                eqty = clean_amt(eqty_row['thstrm_amount'].values[0])
                if eqty > 0:
                    debt_ratio = (liab / eqty) * 100
                    if debt_ratio > 200:
                        result["financial_risk"]["score"] += 30
                        result["financial_risk"]["details"].append(f"부채비율 200% 초과 (현재: {debt_ratio:.1f}%)")

            # 영업이익 증감 확인
            if not op_row.empty:
                curr_op = clean_amt(op_row['thstrm_amount'].values[0])
                prev_op = clean_amt(op_row['frmtrm_amount'].values[0])
                
                if curr_op < 0:
                    result["financial_risk"]["score"] += 40
                    result["financial_risk"]["details"].append("영업이익 적자 발생")
                elif prev_op > 0 and curr_op < prev_op:
                    result["financial_risk"]["score"] += 30
                    result["financial_risk"]["details"].append("영업이익 전년 대비 감소")
    except Exception as e:
        result["financial_risk"]["details"].append(f"재무 데이터 처리 중 오류: {e}")

    # --- 텍스트 공시 리스크 평가 (최근 1년) ---
    try:
        today = datetime.now()
        bgn_de = (today - relativedelta(years=1)).strftime('%Y%m%d')
        end_de = today.strftime('%Y%m%d')

        disclosures = dart_api.list(corp_code, start=bgn_de, end=end_de)
        if disclosures is not None and not disclosures.empty:
            # flr_nm (제출인명)이 기업명과 일치하는 공시만 1차 필터링
            filtered_df = disclosures[disclosures['flr_nm'] == corp_name] 
            
            # report_nm (보고서명) 기반 키워드 탐지
            fatal_keywords = '감자|횡령|배임|상장폐지|부도'
            warning_keywords = '유상증자|소송|해지|생산중단|영업정지'

            fatal_df = filtered_df[filtered_df['report_nm'].str.contains(fatal_keywords, regex=True, na=False)]
            warn_df = filtered_df[filtered_df['report_nm'].str.contains(warning_keywords, regex=True, na=False)]

            if not fatal_df.empty:
                fatal_count = len(fatal_df)
                result["text_risk"]["score"] += (100 * fatal_count)
                result["text_risk"]["details"].append(f"🚨 치명적 악재 공시 {fatal_count}건 발견")
            
            if not warn_df.empty:
                warn_count = len(warn_df)
                result["text_risk"]["score"] += (30 * warn_count)
                result["text_risk"]["details"].append(f"⚠️ 주의성 악재 공시 {warn_count}건 발견")
    except Exception as e:
        result["text_risk"]["details"].append(f"공시 데이터 처리 중 오류: {e}")

    # --- 종합 평가 (Total Score & Grade 산출) ---
    total_score = result["financial_risk"]["score"] + result["text_risk"]["score"]
    result["total_score"] = total_score

    # 점수별 등급 부여 (Rule-based)
    if total_score >= 100:
        result["grade"] = "🚨 위험 (Danger) - 투자 심각한 고려 필요"
    elif total_score >= 40:
        result["grade"] = "⚠️ 주의 (Caution) - 면밀한 모니터링 필요"
    else:
        result["grade"] = "✅ 안전 (Safe) - 특이사항 없음"

    return result

In [13]:
# 함수 테스트
test_risk_result = calculate_total_risk(test_corp_name, test_corp_code, dart)

# 결과 출력
print(json.dumps(test_risk_result, indent=4, ensure_ascii=False))

{
    "corp_name": "삼성전자",
    "total_score": 0,
    "grade": "✅ 안전 (Safe) - 특이사항 없음",
    "financial_risk": {
        "score": 0,
        "details": []
    },
    "text_risk": {
        "score": 0,
        "details": []
    }
}
